In [6]:
!gunzip c4-train.00000-of-01024-30K.json.gz

In [9]:
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

file_path = '/content/c4-train.00000-of-01024-30K.json'
documents = []

with open(file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 30000:
            break
        data = json.loads(line)
        if 'text' in data:
            documents.append(data['text'])
print(f"Length Document: {len(documents)}")

Length Document: 30000


Experiment 1

In [16]:
import nltk
from nltk.util import ngrams
from nltk.probability import FreqDist

In [17]:
nltk.download('punkt')
nltk.download('punkt_tab')

sentence_list = []
for doc in documents:
    sentence_list.extend(nltk.sent_tokenize(doc))
print(f"Number of sentences: {len(sentence_list)}")

word_tokens = []
for s in sentence_list:
    word_tokens.extend(nltk.word_tokenize(s))

total_tokens = len(word_tokens)
print(f"Number of tokens: {total_tokens}")

vocab_set = set(word_tokens)
print(f"Vocabulary size: {len(vocab_set)}")

print(f"Number of unigrams: {total_tokens}")
print(f"Number of bigrams: {max(0, total_tokens - 1)}")
print(f"Number of trigrams: {max(0, total_tokens - 2)}")

freq_uni = FreqDist(ngrams(word_tokens, 1))
freq_bi = FreqDist(ngrams(word_tokens, 2))
freq_tri = FreqDist(ngrams(word_tokens, 3))

unigrams_once = sum(1 for c in freq_uni.values() if c == 1)
bigrams_once = sum(1 for c in freq_bi.values() if c == 1)
trigrams_once = sum(1 for c in freq_tri.values() if c == 1)

print(f"Number of unigrams that appear only once: {unigrams_once}")
print(f"Number of bigrams that appear only once: {bigrams_once}")
print(f"Number of trigrams that appear only once: {trigrams_once}")



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Number of sentences: 619347
Number of tokens: 12446381
Vocabulary size: 314068
Number of unigrams: 12446381
Number of bigrams: 12446380
Number of trigrams: 12446379
Number of unigrams that appear only once: 176375
Number of bigrams that appear only once: 2250263
Number of trigrams that appear only once: 6762045


In [18]:
print("Top 5 unigrams:")
for item, freq in freq_uni.most_common(5):
    print(f"{item[0]}: {freq}")

print("Top 5 bigrams:")
for item, freq in freq_bi.most_common(5):
    print(f"{' '.join(item)}: {freq}")

print("Top 5 trigrams:")
for item, freq in freq_tri.most_common(5):
    print(f"{' '.join(item)}: {freq}")

Top 5 unigrams:
.: 558546
,: 518219
the: 497067
and: 315062
to: 300844
Top 5 bigrams:
of the: 60603
. The: 51838
, and: 46035
in the: 42914
’ s: 41215
Top 5 trigrams:
. If you: 6255
it ’ s: 4885
. It is: 4414
don ’ t: 4329
as well as: 4146


Experiment 2

In [19]:
import random
import math
from collections import Counter
random.seed(42)
random.shuffle(documents)

total_docs = len(documents)
train_idx = int(0.8 * total_docs)
val_idx = int(0.9 * total_docs)

docs_train = documents[:train_idx]
docs_val = documents[train_idx:val_idx]
docs_test = documents[val_idx:]

print(f"Len of train_documents: {len(docs_train)}")
print(f"Len of valid_documents: {len(docs_val)}")
print(f"Len of test_documents: {len(docs_test)}")

def process_tokenization(docs):
    token_data = []
    for d in docs:
        sents = nltk.sent_tokenize(d)
        for s in sents:
            words = nltk.word_tokenize(s)
            if len(words) > 0:
                token_data.append(words)
    return token_data

train_data = process_tokenization(docs_train)
val_data = process_tokenization(docs_val)
test_data = process_tokenization(docs_test)

def insert_boundary_markers(data):
    marked_data = []
    for s in data:
        marked_data.append(["<s>"] + s + ["</s>"])
    return marked_data

train_data = insert_boundary_markers(train_data)
val_data = insert_boundary_markers(val_data)
test_data = insert_boundary_markers(test_data)

def generate_vocab(data):
    vocab = set()
    for s in data:
        for w in s:
            vocab.add(w)
    vocab.add("<UNK>")
    return vocab

vocab_set = generate_vocab(train_data)

def handle_oov_words(data, vocab):
    processed_data = []
    for s in data:
        new_s = []
        for w in s:
            if w in vocab:
                new_s.append(w)
            else:
                new_s.append("<UNK>")
        processed_data.append(new_s)
    return processed_data

train_data = handle_oov_words(train_data, vocab_set)
val_data = handle_oov_words(val_data, vocab_set)
test_data = handle_oov_words(test_data, vocab_set)

def get_unigram_freqs(data):
    freqs = Counter()
    for s in data:
        for w in s:
            freqs[w] += 1
    return freqs

def get_bigram_freqs(data):
    freqs = Counter()
    for s in data:
        for i in range(len(s) - 1):
            freqs[(s[i], s[i + 1])] += 1
    return freqs

def get_trigram_freqs(data):
    freqs = Counter()
    for s in data:
        for i in range(len(s) - 2):
            freqs[(s[i], s[i + 1], s[i + 2])] += 1
    return freqs

uni_freqs = get_unigram_freqs(train_data)
bi_freqs = get_bigram_freqs(train_data)
tri_freqs = get_trigram_freqs(train_data)

def calc_bigram_mle(ctx, w, uni_counts, bi_counts):
    denom = uni_counts[ctx]
    if denom == 0:
        return 0.0
    return bi_counts[(ctx, w)] / denom

def calc_bigram_laplace(ctx, w, uni_counts, bi_counts, vocab):
    v_size = len(vocab)
    num = bi_counts[(ctx, w)] + 1
    denom = uni_counts[ctx] + v_size
    return num / denom

def calc_trigram_mle(ctx, w, bi_counts, tri_counts):
    w1, w2 = ctx
    denom = bi_counts[(w1, w2)]
    if denom == 0:
        return 0.0
    return tri_counts[(w1, w2, w)] / denom

def calc_trigram_laplace(ctx, w, bi_counts, tri_counts, vocab):
    w1, w2 = ctx
    v_size = len(vocab)
    num = tri_counts[(w1, w2, w)] + 1
    denom = bi_counts[(w1, w2)] + v_size
    return num / denom

def compute_sent_prob_bigram(sent, prob_func):
    prob = 1.0
    for i in range(1, len(sent)):
        ctx = sent[i - 1]
        w = sent[i]
        prob *= prob_func(ctx, w)
    return prob

def compute_ppl_bigram(data, prob_func):
    log_prob_sum = 0.0
    total_tokens = 0
    for s in data:
        for i in range(1, len(s)):
            ctx = s[i - 1]
            w = s[i]
            prob = prob_func(ctx, w)
            if prob <= 0:
                return float("inf")
            log_prob_sum += math.log(prob)
            total_tokens += 1
    return math.exp(-log_prob_sum / total_tokens)

def compute_ppl_trigram(data, prob_func):
    log_prob_sum = 0.0
    total_tokens = 0
    for s in data:
        for i in range(2, len(s)):
            ctx = (s[i - 2], s[i - 1])
            w = s[i]
            prob = prob_func(ctx, w)
            if prob <= 0:
                return float("inf")
            log_prob_sum += math.log(prob)
            total_tokens += 1
    return math.exp(-log_prob_sum / total_tokens)

Len of train_documents: 24000
Len of valid_documents: 3000
Len of test_documents: 3000


In [20]:
bi_mle_func = lambda ctx, w: calc_bigram_mle(
    ctx,
    w,
    uni_freqs,
    bi_freqs
)

train_ppl_bi_mle = compute_ppl_bigram(
    train_data,
    bi_mle_func
)

val_ppl_bi_mle = compute_ppl_bigram(
    val_data,
    bi_mle_func
)

test_ppl_bi_mle = compute_ppl_bigram(
    test_data,
    bi_mle_func
)

bi_laplace_func = lambda ctx, w: calc_bigram_laplace(
    ctx,
    w,
    uni_freqs,
    bi_freqs,
    vocab_set
)

train_ppl_bi_laplace = compute_ppl_bigram(
    train_data,
    bi_laplace_func
)

val_ppl_bi_laplace = compute_ppl_bigram(
    val_data,
    bi_laplace_func
)

test_ppl_bi_laplace = compute_ppl_bigram(
    test_data,
    bi_laplace_func
)

tri_mle_func = lambda ctx, w: calc_trigram_mle(
    ctx,
    w,
    bi_freqs,
    tri_freqs
)

train_ppl_tri_mle = compute_ppl_trigram(
    train_data,
    tri_mle_func
)

val_ppl_tri_mle = compute_ppl_trigram(
    val_data,
    tri_mle_func
)

test_ppl_tri_mle = compute_ppl_trigram(
    test_data,
    tri_mle_func
)

tri_laplace_func = lambda ctx, w: calc_trigram_laplace(
    ctx,
    w,
    bi_freqs,
    tri_freqs,
    vocab_set
)

train_ppl_tri_laplace = compute_ppl_trigram(
    train_data,
    tri_laplace_func
)

val_ppl_tri_laplace = compute_ppl_trigram(
    val_data,
    tri_laplace_func
)

test_ppl_tri_laplace = compute_ppl_trigram(
    test_data,
    tri_laplace_func
)

In [21]:
print("\nExperiment 2 — MLE vs Laplace")
print("-" * 60)
print(f"{'Model':<25}{'Train PPL':>12}{'Valid PPL':>12}{'Test PPL':>12}")
print("-" * 60)

print(f"{'Bigram MLE':<25}{train_ppl_bi_mle:>12.4f}{val_ppl_bi_mle:>12.4f}{test_ppl_bi_mle:>12.4f}")

print(f"{'Bigram Laplace':<25}{train_ppl_bi_laplace:>12.4f}{val_ppl_bi_laplace:>12.4f}{test_ppl_bi_laplace:>12.4f}")

print(f"{'Trigram MLE':<25}{train_ppl_tri_mle:>12.4f}{val_ppl_tri_mle:>12.4f}{test_ppl_tri_mle:>12.4f}")

print(f"{'Trigram Laplace':<25}{train_ppl_tri_laplace:>12.4f}{val_ppl_tri_laplace:>12.4f}{test_ppl_tri_laplace:>12.4f}")

print("-" * 60)


Experiment 2 — MLE vs Laplace
------------------------------------------------------------
Model                       Train PPL   Valid PPL    Test PPL
------------------------------------------------------------
Bigram MLE                   118.3204         inf         inf
Bigram Laplace              5486.1551   7076.6080   7078.2423
Trigram MLE                    9.9139         inf         inf
Trigram Laplace            50515.0743  84274.4123  83901.0536
------------------------------------------------------------


# 12. Phân tích và Giải thích thực nghiệm 2

## 12.1. Đánh giá hiện tượng bùng nổ Perplexity trên tập Valid và Test của mô hình MLE

Khi nhìn vào kết quả của Bigram MLE và Trigram MLE, ta thấy Perplexity trên tập Valid và Test đều trả về giá trị vô cực là `inf`.

Nguyên nhân cốt lõi là do vấn đề tần suất bằng không.

Thuật toán MLE tính toán xác suất dựa trên việc đếm tần suất xuất hiện. Khi mô hình dự đoán trên dữ liệu mới, nó chắc chắn sẽ gặp những tổ hợp hai từ là Bigram hoặc ba từ là Trigram chưa từng xuất hiện trong tập huấn luyện.

Lúc này, số lần xuất hiện của cụm từ đó bằng `0`, dẫn đến xác suất N-gram bị MLE gán bằng `0`.

Theo công thức tính Perplexity, hệ thống phải tính logarit tự nhiên của các xác suất này. Khi xác suất bằng `0`:

$$
\log(0) \rightarrow -\infty
$$

Chỉ cần xuất hiện một N-gram có xác suất bằng `0`, tổng log-likelihood của toàn bộ văn bản sẽ trở thành `-\infty`, từ đó khiến Perplexity của toàn bộ văn bản trở thành:

$$
PPL \rightarrow \infty
$$

Vì vậy, việc PPL trên tập Valid và Test của MLE bằng `inf` là hệ quả trực tiếp của việc xuất hiện các N-gram chưa từng có trong tập huấn luyện.

---

## 12.2. Nghịch lý điểm số cực thấp trên tập Training của MLE

Trái ngược với sự thất bại trên tập Test, MLE lại đạt điểm PPL cực kỳ ấn tượng trên tập Training.

Cụ thể:

* Bigram MLE: `PPL = 118.32`
* Trigram MLE: `PPL = 9.91`

Lý do là vì chúng ta đang đánh giá mô hình trên chính bộ dữ liệu mà nó đã được học.

Mọi N-gram đem ra tính toán đều được trích xuất từ tập huấn luyện, đồng nghĩa với việc tần suất đếm của chúng luôn lớn hơn hoặc bằng `1`:

$$
Count(N\text{-gram}) \geq 1
$$

Do đó, không có xác suất nào bằng `0`, nên công thức tính Perplexity vẫn hoạt động bình thường.

Việc Trigram MLE đạt PPL rất thấp với giá trị `9.91` không chứng tỏ đây là một mô hình xuất sắc. Ngược lại, kết quả này cho thấy mô hình đang phụ thuộc rất mạnh vào dữ liệu huấn luyện.

Bậc N-gram càng cao, mô hình càng có khả năng ghi nhớ những chuỗi ngữ cảnh dài hơn. Với Trigram, mô hình có thể ghi nhớ nhiều tổ hợp từ xuất hiện trong tập Training, từ đó tạo ra xác suất rất cao cho những N-gram mà nó đã từng nhìn thấy.

Do đó, kết quả:

> Training PPL thấp nhưng Valid và Test PPL bằng `inf`

cho thấy khả năng tổng quát hóa của mô hình rất kém.

Đây là một biểu hiện rõ ràng của overfitting. Mô hình hoạt động tốt trên dữ liệu đã nhìn thấy nhưng thất bại khi gặp dữ liệu mới.

---

## 12.3. Tác động của Laplace Smoothing

Sự xuất hiện của phương pháp Laplace Smoothing đã giải quyết thành công lỗi PPL vô cực của MLE.

Sau khi áp dụng Laplace Smoothing, Perplexity trên tập Valid và Test không còn trả về `inf` mà trở thành những giá trị hữu hạn.

Ví dụ:

* Bigram Laplace trên Valid: khoảng `7076.60`

Bằng cách cộng thêm `1` vào tần suất của mọi N-gram và cộng kích thước từ vựng vào mẫu số, Laplace Smoothing đảm bảo không có tổ hợp từ nào bị gán xác suất bằng `0`.

Với Bigram:

$$
P(w_i|w_{i-1})
=
\frac{Count(w_{i-1},w_i)+1}
{Count(w_{i-1})+V}
$$

Nhờ phép cộng `1`, ngay cả một N-gram chưa từng xuất hiện trong tập Training cũng có xác suất lớn hơn `0`.

Tuy nhiên, cái giá phải trả là PPL bị đội lên rất cao.

Ví dụ, Trigram Laplace có thể đạt PPL trên `84000`.

Nguyên nhân là do không gian Trigram vô cùng rộng lớn và phân bố rất thưa thớt.

Với một từ vựng có kích thước `V`, số lượng tổ hợp Trigram lý thuyết có thể lên tới:

$$
V^3
$$

Trong khi đó, tập dữ liệu huấn luyện chỉ quan sát được một phần rất nhỏ trong số các tổ hợp này.

Laplace Smoothing lại thực hiện việc cộng `1` một cách đồng đều cho tất cả các tổ hợp, kể cả những Trigram chưa từng xuất hiện.

Điều này khiến một lượng xác suất đáng kể bị phân bổ cho những N-gram không thực sự có ý nghĩa. Kết quả là xác suất dành cho những N-gram thực sự xuất hiện bị giảm xuống, khiến mô hình trở nên kém tự tin hơn và Perplexity tăng rất mạnh.

Có thể tóm tắt kết quả như sau:

| Mô hình         | Training PPL | Valid/Test PPL | Vấn đề chính                          |
| --------------- | -----------: | -------------: | ------------------------------------- |
| Bigram MLE      |     `118.32` |          `inf` | Zero-frequency                        |
| Trigram MLE     |       `9.91` |          `inf` | Zero-frequency và overfitting         |
| Bigram Laplace  |            — |     `~7076.60` | PPL hữu hạn nhưng rất cao             |
| Trigram Laplace |            — |       `>84000` | Làm mịn quá mạnh trên không gian thưa |

### Kết luận

Có thể nhìn thấy một sự đánh đổi rõ ràng:

* MLE không làm thay đổi xác suất của các N-gram đã quan sát, nhưng có thể gán xác suất `0` cho N-gram mới, dẫn đến PPL bằng `inf`.
* Laplace Smoothing loại bỏ hoàn toàn xác suất `0`, giúp mô hình có thể xử lý N-gram chưa từng xuất hiện và đưa PPL về giá trị hữu hạn.
* Tuy nhiên, với N-gram bậc cao như Trigram, Laplace phân bổ quá nhiều xác suất cho những tổ hợp chưa từng xuất hiện, khiến PPL tăng rất mạnh.

Điều này cho thấy việc xử lý vấn đề tần suất bằng không là cần thiết, nhưng cộng `1` một cách đồng đều cho toàn bộ không gian N-gram không phải là giải pháp tối ưu.


In [22]:
test_data = process_tokenization(docs_test)
test_data = insert_boundary_markers(test_data)
test_data = handle_oov_words(test_data, vocab_set)

train_token_count = sum(uni_freqs.values())

def calc_unigram_laplace(w, uni_counts, total_toks, vocab):
    v_size = len(vocab)
    return (uni_counts[w] + 1) / (total_toks + v_size)

uni_laplace_func = lambda w: calc_unigram_laplace(
    w,
    uni_freqs,
    train_token_count,
    vocab_set
)

def compute_ppl_unigram(data, prob_func):
    log_prob_sum = 0.0
    total_tokens = 0
    for s in data:
        for i in range(1, len(s)):
            w = s[i]
            prob = prob_func(w)
            if prob <= 0:
                return float("inf")
            log_prob_sum += math.log(prob)
            total_tokens += 1
    return math.exp(-log_prob_sum / total_tokens)

In [23]:
train_ppl_uni_laplace = compute_ppl_unigram(
    train_data,
    uni_laplace_func
)

val_ppl_uni_laplace = compute_ppl_unigram(
    val_data,
    uni_laplace_func
)

test_ppl_uni_laplace = compute_ppl_unigram(
    test_data,
    uni_laplace_func
)

test_ppl_bi_laplace = compute_ppl_bigram(
    test_data,
    bi_laplace_func
)

test_ppl_tri_laplace = compute_ppl_trigram(
    test_data,
    tri_laplace_func
)

In [24]:
print("\nExperiment 3 — Perplexity Comparison")
print("-" * 65)
print(f"{'Model':<25}{'Train':>12}{'Validation':>14}{'Test':>14}")
print("-" * 65)

print(f"{'Unigram (Laplace)':<25}{train_ppl_uni_laplace:>12.4f}{val_ppl_uni_laplace:>14.4f}{test_ppl_uni_laplace:>14.4f}")
print(f"{'Bigram (Laplace)':<25}{train_ppl_bi_laplace:>12.4f}{val_ppl_bi_laplace:>14.4f}{test_ppl_bi_laplace:>14.4f}")
print(f"{'Trigram (Laplace)':<25}{train_ppl_tri_laplace:>12.4f}{val_ppl_tri_laplace:>14.4f}{test_ppl_tri_laplace:>14.4f}")
print("-" * 65)


Experiment 3 — Perplexity Comparison
-----------------------------------------------------------------
Model                           Train    Validation          Test
-----------------------------------------------------------------
Unigram (Laplace)           1771.2747     1880.5907     1872.7322
Bigram (Laplace)            5486.1551     7076.6080     7078.2423
Trigram (Laplace)          50515.0743    84274.4123    83901.0536
-----------------------------------------------------------------


### 13. Phân tích và Giải thích thực nghiệm 3

#### 13.1. Sự áp đảo của mô hình Unigram khi áp dụng làm mịn

Nhìn vào bảng kết quả, mô hình Unigram cho thấy Perplexity thấp nhất và ổn định nhất trên cả ba tập dữ liệu.

Khi tăng bậc lên Bigram và Trigram, điểm số không những không cải thiện mà còn tăng vọt lên mức hàng nghìn và hàng chục nghìn.

Điều này cho thấy trong điều kiện dữ liệu hiện tại, việc tăng bậc N-gram không mang lại lợi ích rõ rệt khi kết hợp với Laplace Smoothing. Ngược lại, không gian xác suất càng lớn thì mô hình càng gặp nhiều khó khăn trong việc phân bổ xác suất.

---

#### 13.2. Vấn đề không gian thưa thớt của Bigram và Trigram

Sự thất bại của các mô hình bậc cao bắt nguồn từ bản chất thưa thớt của ngôn ngữ.

Khi ghép hai hoặc ba từ lại với nhau, số lượng tổ hợp có thể tạo ra tăng rất nhanh, nhưng phần lớn các tổ hợp này không bao giờ xuất hiện trong dữ liệu thực tế.

Việc áp dụng thuật toán Laplace buộc hệ thống phải cộng thêm một đơn vị tần suất cho tất cả các cụm từ, bao gồm cả những cụm từ chưa từng xuất hiện trong tập huấn luyện.

Hệ quả là một lượng lớn xác suất bị phân bổ cho những cụm từ không xuất hiện trong dữ liệu. Phần xác suất này làm giảm lượng xác suất dành cho những cụm từ thực sự xuất hiện.

Mô hình vì vậy mất đi sự tự tin vào các dự đoán đúng, phân bổ xác suất quá đồng đều và trở nên bối rối hơn. Đây là nguyên nhân khiến Perplexity của Bigram và Trigram tăng lên rất cao.

---

#### 13.3. Lợi thế của Unigram

Ngược lại, mô hình Unigram chỉ đánh giá xác suất của từng từ riêng lẻ nên không gian trạng thái của nó nhỏ hơn rất nhiều, chỉ phụ thuộc vào kích thước tập từ vựng.

Thuật toán Laplace ở mô hình này hoạt động hiệu quả hơn vì không phải phân bổ xác suất cho một lượng lớn tổ hợp từ.

Do đó, xác suất của những từ thực sự xuất hiện trong dữ liệu không bị ảnh hưởng quá mạnh. Mô hình vẫn giữ được khả năng dự đoán tốt hơn so với Bigram và Trigram trong cùng điều kiện.

---

#### 13.4. Giảm hiện tượng quá khớp dữ liệu

Dù điểm số tổng thể của Bigram và Trigram bị đẩy lên cao, thuật toán Laplace đã giúp mô hình ổn định hơn so với MLE.

Khoảng cách Perplexity giữa tập Training, Valid và Test ở từng cấu hình khá đồng đều và không còn sự chênh lệch lớn như trước.

Điều này cho thấy hiện tượng quá khớp dữ liệu đã được giảm đáng kể. Mô hình không còn phụ thuộc quá nhiều vào những N-gram đã xuất hiện trong tập huấn luyện và có khả năng xử lý dữ liệu mới ổn định hơn.

Tuy nhiên, việc Perplexity của Bigram và Trigram vẫn rất cao cho thấy Laplace Smoothing chỉ giải quyết được vấn đề xác suất bằng `0`, chứ chưa giải quyết tốt vấn đề không gian N-gram quá lớn và thưa thớt.

---

### Kết luận

Kết quả thực nghiệm cho thấy Unigram hoạt động ổn định nhất khi kết hợp với Laplace Smoothing trong bộ dữ liệu hiện tại.

Bigram và Trigram có khả năng sử dụng nhiều thông tin ngữ cảnh hơn, nhưng khi áp dụng Laplace Smoothing, không gian xác suất quá lớn khiến xác suất bị phân bổ cho rất nhiều tổ hợp chưa từng xuất hiện.

Vì vậy, việc tăng bậc N-gram không phải lúc nào cũng giúp mô hình ngôn ngữ tốt hơn. Với dữ liệu thưa, phương pháp làm mịn và cách phân bổ xác suất có ảnh hưởng rất lớn đến kết quả Perplexity.

#Application — Next-word prediction

In [25]:
def guess_next_word(input_text, top_n=5):
    raw_tokens = nltk.word_tokenize(input_text)
    safe_tokens = [t if t in vocab_set else "<UNK>" for t in raw_tokens]

    if len(safe_tokens) >= 2:
        ctx1, ctx2 = safe_tokens[-2], safe_tokens[-1]
        options = Counter()

        for (t1, t2, t3), count_val in tri_freqs.items():
            if t1 == ctx1 and t2 == ctx2:
                options[t3] = count_val

        if len(options) > 0:
            sum_val = sum(options.values())
            return [(w, count_val / sum_val) for w, count_val in options.most_common(top_n)]
        else:
            for (b1, b2), count_val in bi_freqs.items():
                if b1 == ctx2:
                    options[b2] = count_val

            if len(options) > 0:
                sum_val = sum(options.values())
                return [(w, count_val / sum_val) for w, count_val in options.most_common(top_n)]
            else:
                sum_val = sum(uni_freqs.values())
                return [(w, count_val / sum_val) for w, count_val in uni_freqs.most_common(top_n)]

    elif len(safe_tokens) == 1:
        ctx1 = safe_tokens[0]
        options = Counter()

        for (b1, b2), count_val in bi_freqs.items():
            if b1 == ctx1:
                options[b2] = count_val

        if len(options) > 0:
            sum_val = sum(options.values())
            return [(w, count_val / sum_val) for w, count_val in options.most_common(top_n)]
        else:
            sum_val = sum(uni_freqs.values())
            return [(w, count_val / sum_val) for w, count_val in uni_freqs.most_common(top_n)]

    else:
        sum_val = sum(uni_freqs.values())
        return [(w, count_val / sum_val) for w, count_val in uni_freqs.most_common(top_n)]

In [26]:
evaluation_samples = [
    ("the cat", "was"),
    ("natural language", "processing"),
    ("machine learning", "models"),
    ("one of the", "most"),
    ("in the", "world")
]

eval_outputs = []
for input_ctx, true_word in evaluation_samples:
    top_guesses = guess_next_word(input_ctx, top_n=3)
    formatted_guesses = ", ".join([f"{w} ({p:.2f})" for w, p in top_guesses])
    eval_outputs.append({
        "Context": input_ctx,
        "Top Predictions": formatted_guesses,
        "Actual next word": true_word
    })

print(f"{'Context':<20}{'Top Predictions':<40}{'Actual Next Word':<15}")
print("-" * 75)
for row in eval_outputs:
    print(f"{row['Context']:<20}{row['Top Predictions']:<40}{row['Actual next word']:<15}")
print("-" * 75)

for input_ctx, true_word in evaluation_samples:
    print(f"Context: {input_ctx}")
    top_guesses = guess_next_word(input_ctx, top_n=3)
    for idx, (w, p) in enumerate(top_guesses, 1):
        print(f"{idx}. {w} {p:.2f}")

Context             Top Predictions                         Actual Next Word
---------------------------------------------------------------------------
the cat             , (0.08), 's (0.06), box (0.06)         was            
natural language    processing (0.43), . (0.43), speech (0.14)processing     
machine learning    and (0.14), , (0.10), to (0.08)         models         
one of the          most (0.02), best (0.01), world (0.01)  most           
in the              world (0.02), past (0.01), first (0.01) world          
---------------------------------------------------------------------------
Context: the cat
1. , 0.08
2. 's 0.06
3. box 0.06
Context: natural language
1. processing 0.43
2. . 0.43
3. speech 0.14
Context: machine learning
1. and 0.14
2. , 0.10
3. to 0.08
Context: one of the
1. most 0.02
2. best 0.01
3. world 0.01
Context: in the
1. world 0.02
2. past 0.01
3. first 0.01


#Application — Sentence ranking

In [27]:
def evaluate_text_continuation(base_text, next_text):
    base_toks = [t if t in vocab_set else "<UNK>" for t in nltk.word_tokenize(base_text)]
    next_toks = [t if t in vocab_set else "<UNK>" for t in nltk.word_tokenize(next_text)]
    merged_toks = base_toks + next_toks

    log_p = 0.0
    for idx in range(len(base_toks), len(merged_toks)):
        if idx >= 2:
            prob_val = tri_laplace_func((merged_toks[idx-2], merged_toks[idx-1]), merged_toks[idx])
        else:
            prob_val = bi_laplace_func(merged_toks[idx-1], merged_toks[idx])
        log_p += math.log(prob_val)

    raw_prob = math.exp(log_p)
    normalized_score = math.exp(log_p / max(1, len(next_toks)))
    return raw_prob, log_p, normalized_score

def sort_text_candidates(base_text, options_list):
    evaluated_list = []
    for opt in options_list:
        raw_prob, log_p, normalized_score = evaluate_text_continuation(base_text, opt)
        evaluated_list.append((opt, raw_prob, log_p, normalized_score))
    evaluated_list.sort(key=lambda item: item[3], reverse=True)
    return evaluated_list

In [28]:
input_ctx_1 = "global warming"
opts_1 = [
    "is a serious environmental issue",
    "potato shoe quickly",
    "causes extreme weather events"
]
sorted_res_1 = sort_text_candidates(input_ctx_1, opts_1)

print(f"Context: '{input_ctx_1}'")
print("-" * 85)
print(f"{'Rank':<6}{'Candidate':<35}{'P(Cand|Context)':<20}{'Log Prob':<12}{'Length-Norm Score':<15}")
print("-" * 85)
for idx, (opt, raw_p, log_p, norm_s) in enumerate(sorted_res_1, 1):
    print(f"{idx:<6}{opt:<35}{raw_p:<20.4e}{log_p:<12.4f}{norm_s:<15.6f}")
print("-" * 85)

input_ctx_2 = "renewable energy"
opts_2 = [
    "sources are essential for the future",
    "window walk yellow behind",
    "adoption requires government support"
]
sorted_res_2 = sort_text_candidates(input_ctx_2, opts_2)

print(f"\nContext: '{input_ctx_2}'")
print("-" * 85)
print(f"{'Rank':<6}{'Candidate':<45}{'P(Cand|Context)':<20}{'Length-Norm Score':<15}")
print("-" * 85)
for idx, (opt, raw_p, log_p, norm_s) in enumerate(sorted_res_2, 1):
    print(f"{idx:<6}{opt:<45}{raw_p:<20.4e}{norm_s:<15.6f}")
print("-" * 85)

Context: 'global warming'
-------------------------------------------------------------------------------------
Rank  Candidate                          P(Cand|Context)     Log Prob    Length-Norm Score
-------------------------------------------------------------------------------------
1     is a serious environmental issue   8.0279e-26          -57.7843    0.000010       
2     causes extreme weather events      1.1036e-21          -48.2557    0.000006       
3     potato shoe quickly                4.9944e-17          -37.5356    0.000004       
-------------------------------------------------------------------------------------

Context: 'renewable energy'
-------------------------------------------------------------------------------------
Rank  Candidate                                    P(Cand|Context)     Length-Norm Score
-------------------------------------------------------------------------------------
1     sources are essential for the future         1.1466e-27       

In [29]:
error_cases = [
    {
        "type": "Correct 1",
        "context": "climate change",
        "expected": "impacts",
        "model_pred": guess_next_word("climate change", top_n=1)[0][0],
        "prob": guess_next_word("climate change", top_n=1)[0][1]
    },
    {
        "type": "Correct 2",
        "context": "one of the",
        "expected": "most",
        "model_pred": guess_next_word("one of the", top_n=1)[0][0],
        "prob": guess_next_word("one of the", top_n=1)[0][1]
    },
    {
        "type": "Incorrect 1",
        "context": "the company",
        "expected": "has",
        "model_pred": guess_next_word("the company", top_n=1)[0][0],
        "prob": guess_next_word("the company", top_n=1)[0][1]
    },
    {
        "type": "Incorrect 2",
        "context": "renewable energy",
        "expected": "sources",
        "model_pred": guess_next_word("renewable energy", top_n=1)[0][0],
        "prob": guess_next_word("renewable energy", top_n=1)[0][1]
    }
]

print(f"{'Category':<15}{'Input Context':<20}{'Prediction':<15}{'Target Word':<15}{'Confidence':<10}")
print("-" * 80)
for item in error_cases:
    print(f"{item['type']:<15}{item['context']:<20}{item['model_pred']:<15}{item['expected']:<15}{item['prob']:<10.4f}")
print("-" * 80)

Category       Input Context       Prediction     Target Word    Confidence
--------------------------------------------------------------------------------
Correct 1      climate change      .              impacts        0.2476    
Correct 2      one of the          most           most           0.0196    
Incorrect 1    the company         ’              has            0.1447    
Incorrect 2    renewable energy    sources        sources        0.1231    
--------------------------------------------------------------------------------


### 22. Đánh giá và Phân tích nguyên nhân lỗi

#### 22.1. Nhóm các trường hợp dự đoán chính xác

**Trường hợp 1: Dự đoán trúng thuật ngữ có tính liên kết chặt chẽ**

- Ngữ cảnh đầu vào: `renewable energy`
- Từ mô hình dự đoán: `sources`
- Từ kỳ vọng: `sources`
- Độ tự tin: `0.1231`
- Phân tích nguyên nhân: Cụm từ năng lượng tái tạo đi kèm với chữ nguồn là một thuật ngữ chuẩn hóa xuất hiện với tần suất cao trong dữ liệu huấn luyện. Nhờ sự xuất hiện lặp lại của tổ hợp này, mô hình có nhiều cơ sở thống kê để ưu tiên từ `sources`. Điều này giúp hệ thống đưa ra dự đoán chính xác.

**Trường hợp 2: Dự đoán đúng khuôn mẫu ngữ pháp cố định**

- Ngữ cảnh đầu vào: `one of the`
- Từ mô hình dự đoán: `most`
- Từ kỳ vọng: `most`
- Độ tự tin: `0.0196`
- Phân tích nguyên nhân: Đây là một cấu trúc ngữ pháp phổ biến trong tiếng Anh. Khi gặp chuỗi `one of the`, mô hình có thể dựa vào tần suất xuất hiện trong dữ liệu để nhận biết các từ thường đứng tiếp theo. Trong trường hợp này, `most` có xác suất cao hơn các lựa chọn khác nên mô hình đưa ra dự đoán chính xác.

---

#### 22.2. Nhóm các trường hợp dự đoán sai

**Trường hợp 3: Sai do thói quen ngắt câu và tác dụng phụ của bước tiền xử lý**

- Ngữ cảnh đầu vào: `climate change`
- Từ mô hình dự đoán: `.`
- Từ kỳ vọng: `impacts`
- Độ tự tin: `0.2476`
- Phân tích nguyên nhân: Trong dữ liệu văn bản, cụm `climate change` có thể xuất hiện ở nhiều vị trí khác nhau, trong đó có những trường hợp đứng gần cuối câu. Do dấu câu được giữ lại như một token riêng, dấu chấm có thể xuất hiện với tần suất cao sau cụm từ này. Vì vậy, mô hình ưu tiên dấu chấm thay vì từ `impacts`.

**Trường hợp 4: Sai do ảnh hưởng của dạng sở hữu cách**

- Ngữ cảnh đầu vào: `the company`
- Từ mô hình dự đoán: `'`
- Từ kỳ vọng: `has`
- Độ tự tin: `0.1447`
- Phân tích nguyên nhân: Lỗi này xuất phát từ cách hệ thống xử lý dấu nháy trong quá trình tách từ. Trong dữ liệu tiếng Anh, dấu nháy có thể xuất hiện trong nhiều cấu trúc khác nhau, đặc biệt là các dạng sở hữu cách. Vì vậy, mô hình có thể học được mối liên hệ khá mạnh giữa danh từ và dấu nháy. Với ngữ cảnh chỉ gồm hai từ `the company`, thông tin chưa đủ để mô hình xác định chắc chắn rằng từ tiếp theo phải là một động từ như `has`, dẫn đến việc lựa chọn dấu nháy.

#Context Length

In [30]:
val_uni_total = sum(len(seq) - 1 for seq in val_data)
val_bi_total = sum(len(seq) - 1 for seq in val_data)
val_tri_total = sum(max(0, len(seq) - 2) for seq in val_data)

val_uni_unseen = sum(1 for seq in val_data for idx in range(1, len(seq)) if seq[idx] == "<UNK>")
val_bi_unseen = sum(1 for seq in val_data for idx in range(1, len(seq)) if bi_freqs[(seq[idx-1], seq[idx])] == 0)
val_tri_unseen = sum(1 for seq in val_data for idx in range(2, len(seq)) if tri_freqs[(seq[idx-2], seq[idx-1], seq[idx])] == 0)

print(f"{'Order':<15}{'Train N-grams':>18}{'Total Valid Tokens':>22}{'Unseen in Valid':>18}{'Unseen Ratio (%)':>18}")
print("-" * 95)
print(f"{'Unigram (N=1)':<15}{len(uni_freqs):>18}{val_uni_total:>22}{val_uni_unseen:>18}{val_uni_unseen / val_uni_total * 100:>17.2f}%")
print(f"{'Bigram (N=2)':<15}{len(bi_freqs):>18}{val_bi_total:>22}{val_bi_unseen:>18}{val_bi_unseen / val_bi_total * 100:>17.2f}%")
print(f"{'Trigram (N=3)':<15}{len(tri_freqs):>18}{val_tri_total:>22}{val_tri_unseen:>18}{val_tri_unseen / val_tri_total * 100:>17.2f}%")
print("-" * 95)

Order               Train N-grams    Total Valid Tokens   Unseen in Valid  Unseen Ratio (%)
-----------------------------------------------------------------------------------------------
Unigram (N=1)              271516               1260638             31393             2.49%
Bigram (N=2)              2665369               1260638            276447            21.93%
Trigram (N=3)             6230572               1200160            696632            58.04%
-----------------------------------------------------------------------------------------------


### 23. Đánh giá ảnh hưởng của độ dài ngữ cảnh

#### 23.1. Ngữ cảnh dài hơn không phải lúc nào cũng tốt hơn

Thông qua các số liệu thống kê, có thể thấy việc tăng độ dài ngữ cảnh không đồng nghĩa với việc mô hình sẽ hoạt động hiệu quả hơn.

Trong các mô hình ngôn ngữ truyền thống, khi ngữ cảnh dài hơn, số lượng tổ hợp từ cần xử lý cũng tăng lên rất nhanh. Nếu dữ liệu không đủ lớn, mô hình sẽ khó ước lượng chính xác xác suất của những tổ hợp này.

Vì vậy, việc tăng từ Unigram lên Bigram và Trigram trong trường hợp này không giúp cải thiện kết quả mà ngược lại còn làm Perplexity tăng mạnh.

---

#### 23.2. Sự gia tăng nhanh của số lượng N-gram

Nhìn vào số lượng N-gram trên tập huấn luyện, có thể thấy sự gia tăng rất lớn khi tăng bậc của mô hình.

Không gian từ vựng của Unigram ban đầu chỉ ở mức hơn 271 nghìn. Khi chuyển sang Bigram, số lượng N-gram tăng lên hơn 2.6 triệu và với Trigram là khoảng 6.2 triệu.

Số lượng N-gram tăng nhanh khiến mô hình phải xử lý một lượng lớn các tổ hợp từ khác nhau. Trong khi đó, nhiều tổ hợp chỉ xuất hiện một vài lần hoặc thậm chí không xuất hiện trong tập huấn luyện.

Điều này khiến mô hình dễ ghi nhớ các mẫu đã xuất hiện trong dữ liệu nhưng gặp khó khăn khi xử lý những tổ hợp mới.

---

#### 23.3. Dữ liệu thưa và tỷ lệ N-gram chưa từng xuất hiện

Cột tỷ lệ chưa từng xuất hiện cho thấy sự khác biệt rất lớn giữa các bậc N-gram.

Ở cấu hình Unigram, khi gặp dữ liệu mới, hệ thống chỉ có khoảng `2.49%` từ chưa từng xuất hiện trong tập huấn luyện.

Tuy nhiên, khi chuyển sang Trigram, tỷ lệ này tăng lên `58.04%`.

Điều đó có nghĩa là gần `60%` số Trigram xuất hiện trên tập Valid chưa từng xuất hiện trong quá trình huấn luyện.

Nguyên nhân là do ngôn ngữ tự nhiên có rất nhiều cách kết hợp từ khác nhau. Khi xét từng từ riêng lẻ, số lượng từ mới không quá lớn. Nhưng khi kết hợp thành hai hoặc ba từ, số lượng tổ hợp tăng lên rất nhanh và dữ liệu trở nên thưa hơn.

---

#### 23.4. Ảnh hưởng đến Perplexity

Những số liệu trên giải thích khá rõ kết quả Perplexity ở các thực nghiệm trước.

Trên tập huấn luyện, Trigram có Perplexity rất thấp vì phần lớn các cụm từ mà mô hình gặp đã xuất hiện trong dữ liệu huấn luyện. Mô hình có thể sử dụng trực tiếp các thống kê đã học để đưa ra xác suất cao cho những cụm từ này.

Tuy nhiên, khi chuyển sang tập Valid và Test, có tới `58.04%` Trigram chưa từng xuất hiện. Với MLE, chỉ cần một Trigram có tần suất bằng `0` thì xác suất của nó cũng bằng `0`, từ đó làm Perplexity trở thành `inf`.

Khi sử dụng Laplace Smoothing, vấn đề xác suất bằng `0` được giải quyết. Tuy nhiên, mô hình phải phân bổ thêm xác suất cho một lượng rất lớn các Trigram chưa từng xuất hiện. Điều này làm giảm xác suất dành cho những cụm từ thực sự xuất hiện và khiến Perplexity của Trigram tăng lên hơn `84000`.

Trong khi đó, Unigram có tỷ lệ từ chưa từng xuất hiện thấp hơn rất nhiều nên việc ước lượng xác suất ổn định hơn. Đây cũng là lý do Unigram có Perplexity thấp và ít chênh lệch giữa các tập dữ liệu.

### Kết luận

Kết quả thực nghiệm cho thấy tăng độ dài ngữ cảnh không phải lúc nào cũng mang lại kết quả tốt hơn.

Bigram và Trigram cung cấp thêm thông tin về các từ đứng trước, nhưng đồng thời cũng làm số lượng tổ hợp tăng rất nhanh. Khi dữ liệu huấn luyện chưa đủ lớn, điều này dẫn đến tình trạng nhiều N-gram chưa từng xuất hiện và làm cho việc ước lượng xác suất trở nên khó khăn.

Trong bộ dữ liệu này, Unigram cho kết quả ổn định hơn vì không gian mô hình nhỏ và tỷ lệ từ chưa từng xuất hiện thấp. Ngược lại, Trigram gặp nhiều vấn đề hơn do số lượng tổ hợp lớn và dữ liệu thưa.

In [32]:
import pandas as pd

ppl_data = {
    "Model": [
        "Bigram MLE", "Bigram Laplace",
        "Trigram MLE", "Trigram Laplace",
        "Unigram Laplace"
    ],
    "Train PPL": [
        train_ppl_bi_mle, train_ppl_bi_laplace,
        train_ppl_tri_mle, train_ppl_tri_laplace,
        train_ppl_uni_laplace
    ],
    "Valid PPL": [
        val_ppl_bi_mle, val_ppl_bi_laplace,
        val_ppl_tri_mle, val_ppl_tri_laplace,
        val_ppl_uni_laplace
    ],
    "Test PPL": [
        test_ppl_bi_mle, test_ppl_bi_laplace,
        test_ppl_tri_mle, test_ppl_tri_laplace,
        test_ppl_uni_laplace
    ]
}
df_ppl = pd.DataFrame(ppl_data)

df_pred = pd.DataFrame(eval_outputs)
df_pred.rename(columns={"Actual next word": "Actual Next Word"}, inplace=True)

with open('results.csv', 'w', encoding='utf-8') as f:
    f.write("EXPERIMENT 2 & 3: PERPLEXITY RESULTS\n")
    df_ppl.to_csv(f, index=False)

    f.write("\n\nAPPLICATION: NEXT-WORD PREDICTION\n")
    df_pred.to_csv(f, index=False)

print("Created results.csv")

Created results.csv
